# MLP MaxAD Statistics

Follow-up to `mlp_ablations.ipynb`'s `stat_descriptors` section (explicit per-instance statistics instead of histograms): tests `maxAD`, max absolute deviation from an instance's own median (`max(|x - median(x)|)`), as a replacement for `mean`/`std`, not an addition on top of them. Full writeup: `MLP_MaxAD_Findings.md`.

## Setup

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "scripts"))
from build_points_table import build_and_save_points_table
from taxonomy_separability import add_relative_features
from mlp_classifier import apply_mlp_class_groups
from mlp_variants import run_variant

import pandas as pd

In [2]:
df = build_and_save_points_table()
df = add_relative_features(df)
df = apply_mlp_class_groups(df)

/home/bruno/radar_machine_learning_ws/results/data/points_table.parquet already covers exactly the requested sequences, skipping build


Baseline reference (cached, no retraining), both variants below are measured against this.

In [3]:
_, _, baseline_metrics, _, _ = run_variant(df, "baseline")
baseline_metrics

/home/bruno/radar_machine_learning_ws/results/mlp/mlp_training_history.json already matches this config, loading cached model + history


Saved /home/bruno/radar_machine_learning_ws/results/mlp/mlp_training_curves.png


/home/bruno/radar_machine_learning_ws/results/mlp/mlp_val_metrics.json already cached, reusing
per-class precision/recall/f1 (val):
                  precision  recall     f1  support
car                   0.937   0.784  0.854    31055
large_vehicle         0.719   0.797  0.756     6247
two_wheeler           0.475   0.545  0.508     4629
pedestrian            0.534   0.944  0.682    11325
pedestrian_group      0.754   0.544  0.632    17858
Saved /home/bruno/radar_machine_learning_ws/results/mlp/mlp_confusion_matrix.png


Saved /home/bruno/radar_machine_learning_ws/results/mlp/mlp_precision_recall_f1.png


,precision,recall,f1,support
car,0.937115,0.784093,0.853802,31055
large_vehicle,0.719491,0.796542,0.756059,6247
two_wheeler,0.475302,0.544610,0.507601,4629
pedestrian,0.533806,0.943929,0.681956,11325
pedestrian_group,0.754346,0.544238,0.632295,17858


## 1. `maxad_stats`: swapped in for `stat_descriptors`'s existing stats

Per-feature swap, not a uniform rule: wherever `stat_descriptors` had `mean`+`median`, drop `mean` (redundant central tendency) and keep `median`; wherever it had `std`, replace with `maxAD`; nowhere is a stat added where none existed before.

| feature | `stat_descriptors` | `maxad_stats` |
|---|---|---|
| `rcs` | mean, median, std | median, maxAD |
| `vr_compensated` | mean, median | median |
| `radial` | std | maxAD |
| `azimuth_sc` | std | maxAD |
| `doppler_spread` | as-is (unbinned) | as-is (unbinned) |

50 epochs (baseline default). Defined in `scripts/mlp_variants.py`'s `maxad_stats`, results at `results/mlp/maxad_stats/`.

In [4]:
# cached, no retraining
_, _, maxad_stats_metrics, _, _ = run_variant(df, "maxad_stats")
pd.DataFrame({
    "baseline": baseline_metrics["f1"],
    "maxad_stats": maxad_stats_metrics["f1"],
    "delta": maxad_stats_metrics["f1"] - baseline_metrics["f1"],
})

/home/bruno/radar_machine_learning_ws/results/mlp/maxad_stats/mlp_training_history.json already matches this config, loading cached model + history
Saved /home/bruno/radar_machine_learning_ws/results/mlp/maxad_stats/mlp_training_curves.png


/home/bruno/radar_machine_learning_ws/results/mlp/maxad_stats/mlp_val_metrics.json already cached, reusing
per-class precision/recall/f1 (val):
                  precision  recall     f1  support
car                   0.940   0.718  0.814    31055
large_vehicle         0.583   0.794  0.672     6247
two_wheeler           0.468   0.575  0.516     4629
pedestrian            0.506   0.780  0.614    11325
pedestrian_group      0.675   0.594  0.632    17858
Saved /home/bruno/radar_machine_learning_ws/results/mlp/maxad_stats/mlp_confusion_matrix.png


Saved /home/bruno/radar_machine_learning_ws/results/mlp/maxad_stats/mlp_precision_recall_f1.png


,baseline,maxad_stats,delta
car,0.853802,0.814406,-0.039395
large_vehicle,0.756059,0.672223,-0.083835
two_wheeler,0.507601,0.516092,0.008491
pedestrian,0.681956,0.613515,-0.068441
pedestrian_group,0.632295,0.631939,-0.000355


Macro F1: `baseline` 0.686, `maxad_stats` 0.650 (-0.036), driven by `large_vehicle` (-0.084) and `pedestrian` (-0.068), `two_wheeler`/`pedestrian_group` roughly flat.

**Finding:** a real-looking drop, concentrated in exactly the two classes where central tendency (mean/median of `rcs`/`vr_compensated`) plausibly carries more separating signal than an outlier-driven deviation stat does. Caveat: 0.650 sits right at the bottom edge of `baseline`'s own split-choice noise floor (0.651-0.734, `split_sensitivity.py`), and that floor isn't guaranteed to transfer to a different feature set. Not validated against its own noise floor, only compared to `baseline`'s.

## 2. `maxad_stats_5features`: median + maxAD applied uniformly, 5 point-level features, 200 epochs

Broader version: `{median, maxAD}` uniformly on `x_rel`, `y_rel`, `vr_compensated`, `range_sc`, `rcs` (the baseline histogram's own 4 point-level features plus `range_sc`, dropping `radial`/`azimuth_sc`), `doppler_spread` unbinned as usual. Epochs raised from 50 to 200 to rule out undertraining as a factor in `maxad_stats`'s drop; training curves had flattened by epoch 200.

Defined in `scripts/mlp_variants.py`'s `maxad_stats_5features`, results at `results/mlp/maxad_stats_5features/`.

In [5]:
# cached, no retraining
_, _, maxad_uniform_metrics, _, _ = run_variant(df, "maxad_stats_5features")
pd.DataFrame({
    "baseline": baseline_metrics["f1"],
    "maxad_stats": maxad_stats_metrics["f1"],
    "maxad_stats_5features": maxad_uniform_metrics["f1"],
    "delta vs baseline": maxad_uniform_metrics["f1"] - baseline_metrics["f1"],
})

/home/bruno/radar_machine_learning_ws/results/mlp/maxad_stats_5features/mlp_training_history.json already matches this config, loading cached model + history
Saved /home/bruno/radar_machine_learning_ws/results/mlp/maxad_stats_5features/mlp_training_curves.png


/home/bruno/radar_machine_learning_ws/results/mlp/maxad_stats_5features/mlp_val_metrics.json already cached, reusing
per-class precision/recall/f1 (val):
                  precision  recall     f1  support
car                   0.941   0.757  0.839    31055
large_vehicle         0.630   0.802  0.706     6247
two_wheeler           0.529   0.675  0.593     4629
pedestrian            0.585   0.859  0.696    11325
pedestrian_group      0.744   0.651  0.695    17858
Saved /home/bruno/radar_machine_learning_ws/results/mlp/maxad_stats_5features/mlp_confusion_matrix.png


Saved /home/bruno/radar_machine_learning_ws/results/mlp/maxad_stats_5features/mlp_precision_recall_f1.png


,baseline,maxad_stats,maxad_stats_5features,delta vs baseline
car,0.853802,0.814406,0.838962,-0.014840
large_vehicle,0.756059,0.672223,0.705799,-0.050259
two_wheeler,0.507601,0.516092,0.593430,0.085829
pedestrian,0.681956,0.613515,0.696084,0.014128
pedestrian_group,0.632295,0.631939,0.694580,0.062285


Macro F1: `baseline` 0.686, `maxad_stats` 0.650, `maxad_stats_5features` 0.706 (+0.020 vs baseline). Gain concentrated in `two_wheeler` (+0.085) and `pedestrian_group` (+0.063), at the cost of `large_vehicle` (-0.050) and a smaller `car` cost (-0.015); `pedestrian` roughly flat (+0.014). Same shape as `combined_features`'s validated gain (concentrated in `two_wheeler`/`pedestrian_group`), except here `large_vehicle` is traded away rather than untouched.

**Finding:** promising, but not yet a validated result. +0.020 on a single split is the same order of magnitude as `combined_features`'s initial single-split delta (+0.027), which only became a validated finding after scoring across all 6 `split_sensitivity` folds (mean +0.036, wins every fold) - the single-split number alone sits well inside `baseline`'s 0.651-0.734 split-choice noise floor and isn't distinguishable from noise on its own. Same next step applies here before this counts as a real gain: run `split_sensitivity.run_split_sensitivity` with this feature set across the 6 folds.

## Open question

Neither variant above has been checked against a feature-set-specific split-choice noise floor. `maxad_stats_5features`'s +0.020 is the more interesting number (positive, plausible mechanism, matches `combined_features`'s established gain pattern), but per this project's own standing bar for calling something a real result, it needs the 6-fold validation before being treated as one.